# Mixed Random Regret Minimisation — Standalone Fit and Search

Mixed-RRM combines:
- **Regret minimisation**: choice probabilities are based on regrets,
  not utilities
- **Random parameters**: attribute weights vary across individuals
  (taste heterogeneity), approximated via simulation

This model is appropriate when you believe:
1. Travellers avoid regret rather than maximise utility, AND
2. The strength of regret avoidance varies across individuals

MLE is JAX-accelerated.

In [ ]:
!pip install SearchLibrium --upgrade -q

In [ ]:
import numpy as np
import pandas as pd
from SearchLibrium import MixedRandomRegret, Parameters, call_siman

## 1. Load data

In [ ]:
url = 'https://raw.githubusercontent.com/zahern/HypothesisX/refs/heads/main/data/Swissmetro_final.csv'
df  = pd.read_csv(url)
print(df.columns.tolist())
print(df.head())

## 2. Standalone Mixed-RRM fit

`MixedRandomRegret` inherits from both `RandomRegret` and `MixedLogit`,
so it accepts the same constructor arguments as `MixedLogit` plus the
RRM-specific dataframe format.

In [ ]:
mrrm = MixedRandomRegret(df=df)
mrrm.fit()
print('Log-likelihood:', mrrm.loglik)
print('BIC:           ', mrrm.bic)

## 3. Search over Mixed-RRM specifications

The search explores:
- Which attributes to include
- Which attributes should have random (heterogeneous) weights
- Which distribution family for each random attribute

Set `allow_random=True` — this is **required** for Mixed-RRM.

In [ ]:
varnames   = ['TIME', 'COST', 'HEADWAY', 'SEATS']
choice_set = np.unique(df['alt']).tolist()

params = Parameters(
    criterions   = [('bic', -1)],
    df           = df,
    varnames     = varnames,
    asvarnames   = varnames,
    isvarnames   = [],
    choice_set   = choice_set,
    choices      = df['CHOICE'].values,
    alt_var      = df['alt'].values,
    choice_id    = df['custom_id'].values,
    ind_id       = df['ID'].values,
    base_alt     = 'SM',
    models       = ['mixed_random_regret'],
    allow_random = True,           # REQUIRED for Mixed-RRM
    n_draws      = 500,
    p_val        = 0.05,
    all_sig      = True,
)

best = call_siman(params, init_sol=None, id_num=1,
                  ctrl=(200, 0.001, 50, 10))

## 4. Full model comparison: MNL vs MXL vs RRM vs Mixed-RRM

Let the search decide which model class fits best.

In [ ]:
params_all = Parameters(
    criterions   = [('bic', -1)],
    df           = df,
    varnames     = varnames,
    asvarnames   = varnames,
    isvarnames   = [],
    choice_set   = choice_set,
    choices      = df['CHOICE'].values,
    alt_var      = df['alt'].values,
    choice_id    = df['custom_id'].values,
    ind_id       = df['ID'].values,
    base_alt     = 'SM',
    models       = ['multinomial', 'mixed_logit',
                    'random_regret', 'mixed_random_regret'],
    allow_random = True,
    n_draws      = 300,
    p_val        = 0.05,
)

best_all = call_siman(params_all, init_sol=None, id_num=2,
                      ctrl=(500, 0.001, 100, 20))
print('Winner model:', best_all.get('model_n'))
print('BIC:         ', best_all.get('bic'))
print('Variables:   ', best_all.get('asvars'))
print('Random:      ', best_all.get('randvars'))